In [1]:
!nvidia-smi

Tue Mar  3 13:02:17 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.126.16             Driver Version: 580.126.16     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L40S                    Off |   00000000:84:00.0 Off |                    0 |
| N/A   26C    P8             33W /  350W |       0MiB /  46068MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!conda list

^C



KeyboardInterrupt



In [3]:
import torch
from torch.utils.data import DataLoader
import argparse
import os
from tqdm import tqdm
from datasets import load_from_disk

In [4]:
from src.model.CNN import ChexpertCNN
from src.model.CLIP import ChexpertCLIP
from src.model.transformer import ChexpertTransformer

In [5]:
def collate_fn(batch):
    images = torch.stack([b["image_u8"] for b in batch])   # uint8 CHW
    labels = torch.stack([b["labels"] for b in batch])
    return images, labels

In [14]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DEVICE

device(type='cuda')

In [16]:
print("loading cnn...")
cnn_model = ChexpertCNN().to(DEVICE)
cnn_model.load_state_dict(torch.load("models/init/cnn/cnn_chexpert.pth"))

print("loading clip...")
clip_model = ChexpertCLIP().to(DEVICE)
clip_model.load_state_dict(torch.load("models/init/clip/clip_chexpert.pth"))

print("loading transformer...")
transformer_model = ChexpertTransformer().to(DEVICE)
transformer_model.load_state_dict(torch.load("models/init/transformer/transformer_chexpert.pth"))

loading cnn...
loading clip...


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
vision_model.embeddings.position_ids | UNEXPECTED |  | 
text_model.embeddings.position_ids   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


loading transformer...


<All keys matched successfully>

In [20]:
!pip install matplotlib

  Using cached matplotlib-3.10.8-cp314-cp314-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (52 kB)
  Using cached contourpy-1.3.3-cp314-cp314-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (5.5 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
Using cached matplotlib-3.10.8-cp314-cp314-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (9.8 MB)
Using cached contourpy-1.3.3-cp314-cp314-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (363 kB)
Using cached cycler-0.12.1-py3-none-any.whl (8.3 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [matplotlib]━━━━━━━ 2/3 [matplotlib]


In [21]:
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from pathlib import Path

In [22]:
def compute_saliency_map(model, images, labels, device):
    """
    Compute saliency maps for a batch of images.
    
    Args:
        model: The neural network model
        images: Input images (B, C, H, W) - already normalized
        labels: Ground truth labels (B, num_classes)
        device: Device to use
        
    Returns:
        saliency_maps: (B, H, W) - max absolute gradient across channels
    """
    # Enable gradient computation for input
    images.requires_grad_(True)
    
    # Forward pass
    outputs = model(images)
    
    # Compute loss for each sample and get gradients
    # Use binary cross entropy with logits
    loss = torch.nn.functional.binary_cross_entropy_with_logits(
        outputs, labels.clamp(min=0.0), reduction='none'
    )
    
    # Sum loss across classes (to get total loss per sample)
    loss_per_sample = loss.sum(dim=1)
    total_loss = loss_per_sample.mean()
    
    # Backward pass
    total_loss.backward()
    
    # Get gradients with respect to input
    gradients = images.grad.data  # (B, C, H, W)
    
    # Compute saliency map as max absolute gradient across channels
    saliency_maps = torch.max(torch.abs(gradients), dim=1)[0]  # (B, H, W)
    
    return saliency_maps.cpu().numpy()

def normalize_saliency(saliency_map):
    """Normalize saliency map to [0, 1] range."""
    min_val = saliency_map.min()
    max_val = saliency_map.max()
    if max_val > min_val:
        return (saliency_map - min_val) / (max_val - min_val)
    else:
        return saliency_map

def visualize_and_save_saliency(original_image, saliency_map, output_path, colormap='jet'):
    """
    Visualize saliency map and save as image.
    
    Args:
        original_image: Original image (C, H, W) in range [0, 1] or [0, 255]
        saliency_map: Saliency map (H, W) in range [0, 1]
        output_path: Path to save the visualization
        colormap: Colormap to use for visualization
    """
    # Ensure output directory exists
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    
    # Normalize saliency_map to [0, 1]
    saliency_normalized = normalize_saliency(saliency_map)
    
    # Create figure with subplots: original + saliency + overlay
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    # Plot original image
    if original_image.shape[0] == 3:
        # RGB image
        img_display = original_image.transpose(1, 2, 0)
        img_display = np.clip(img_display, 0, 1)
        axes[0].imshow(img_display, cmap='gray')
    else:
        # Grayscale image
        axes[0].imshow(original_image[0], cmap='gray')
    axes[0].set_title('Original Image')
    axes[0].axis('off')
    
    # Plot saliency map
    im1 = axes[1].imshow(saliency_normalized, cmap=colormap)
    axes[1].set_title('Saliency Map')
    axes[1].axis('off')
    plt.colorbar(im1, ax=axes[1])
    
    # Plot overlay (original + saliency)
    if original_image.shape[0] == 3:
        img_display = original_image.transpose(1, 2, 0)
        img_display = np.clip(img_display, 0, 1)
        axes[2].imshow(img_display, cmap='gray', alpha=0.6)
    else:
        axes[2].imshow(original_image[0], cmap='gray', alpha=0.6)
    im2 = axes[2].imshow(saliency_normalized, cmap=colormap, alpha=0.6)
    axes[2].set_title('Overlay')
    axes[2].axis('off')
    plt.colorbar(im2, ax=axes[2])
    
    plt.tight_layout()
    plt.savefig(output_path, dpi=100, bbox_inches='tight')
    plt.close()

def load_jpeg_image(image_path, target_size=(224, 224)):
    """
    Load a JPEG image and convert to tensor format matching model input expectations.
    
    Args:
        image_path: Path to JPEG file
        target_size: Target size (H, W) for resizing
        
    Returns:
        image_tensor: (1, 3, H, W) tensor in [0, 255] range (uint8 equivalent)
        original_image: (3, H, W) numpy array in [0, 1] range for visualization
    """
    # Load image
    img = Image.open(image_path).convert('RGB')
    
    # Resize to target size
    img = img.resize(target_size, Image.Resampling.LANCZOS)
    
    # Convert to numpy array
    img_np = np.array(img, dtype=np.uint8)  # (H, W, 3)
    
    # Convert to tensor format: (1, 3, H, W)
    image_tensor = torch.from_numpy(img_np).permute(2, 0, 1).unsqueeze(0)  # (1, 3, H, W)
    
    # Also prepare for visualization: (3, H, W) in [0, 1] range
    original_image = img_np.astype(np.float32).transpose(2, 0, 1) / 255.0
    
    return image_tensor, original_image

In [26]:
# Example 1: Generate saliency map from a single JPEG image
image_path = "chexlocalize/chexlocalize/CheXpert/test/patient64745/study1/view1_frontal.jpg"  # Replace with path to your image

# if os.path.exists(image_path):
print(f"Loading image: {image_path}")
image_tensor, original_image = load_jpeg_image(image_path)

# Normalize
print("Normalizing image")
image_normalized = image_tensor.to(DEVICE).float().div_(255.0)
mean = torch.tensor([0.485, 0.456, 0.406], device=DEVICE).view(1, 3, 1, 1)
std = torch.tensor([0.229, 0.224, 0.225], device=DEVICE).view(1, 3, 1, 1)
image_normalized = (image_normalized - mean) / std

# Create dummy labels
labels = torch.zeros(1, 5, device=DEVICE)

Loading image: chexlocalize/chexlocalize/CheXpert/test/patient64745/study1/view1_frontal.jpg
Normalizing image


In [27]:
# Compute saliency maps for each model
for model_name, model in [("CNN", cnn_model), ("Transformer", transformer_model)]:
    print(f"\nGenerating saliency map for {model_name}...")
    model.eval()
    
    with torch.enable_grad():
        saliency_map = compute_saliency_map(model, image_normalized, labels, DEVICE)
    
    # Save visualization
    
    output_path = f"maps/saliency2_{model_name}.png"
    print("saving to", output_path)
    visualize_and_save_saliency(original_image, saliency_map[0], output_path)
    print(f"✓ Saved to {output_path}")
# else:
#     print(f"Image file not found: {image_path}")


Generating saliency map for CNN...
saving to maps/saliency2_CNN.png
✓ Saved to maps/saliency2_CNN.png

Generating saliency map for Transformer...
saving to maps/saliency2_Transformer.png
✓ Saved to maps/saliency2_Transformer.png


In [ ]:
# Example 2: Generate saliency maps from validation dataset
# This processes multiple images from the validation set and saves them

num_samples = 10  # Number of samples to process
output_base_dir = "saliency_maps_notebook"

# Load validation dataset
print("Loading validation dataset...")
validation_dataset = load_from_disk("data/validation")
validation_dataset.set_format(type="torch", columns=["image_u8", "labels"])

# Create dataloader
validation_dataloader = DataLoader(
    validation_dataset,
    batch_size=4,
    shuffle=False,
    num_workers=4,
    pin_memory=True,
    collate_fn=collate_fn,
)

# Process CNN and Transformer models
for model_name, model in [("cnn", cnn_model), ("transformer", transformer_model)]:
    print(f"\n{'='*60}")
    print(f"Processing {model_name.upper()} model")
    print(f"{'='*60}")
    
    model.eval()
    model_output_dir = os.path.join(output_base_dir, model_name)
    os.makedirs(model_output_dir, exist_ok=True)
    
    sample_count = 0
    for images, labels in validation_dataloader:
        # Normalize images
        images_normalized = images.to(DEVICE).float().div_(255.0)
        mean = torch.tensor([0.485, 0.456, 0.406], device=DEVICE).view(1, 3, 1, 1)
        std = torch.tensor([0.229, 0.224, 0.225], device=DEVICE).view(1, 3, 1, 1)
        images_normalized = (images_normalized - mean) / std
        
        labels = labels.to(DEVICE)
        
        # Compute saliency maps
        with torch.enable_grad():
            saliency_maps = compute_saliency_map(model, images_normalized, labels, DEVICE)
        
        # Save visualizations
        batch_size = images.shape[0]
        for i in range(batch_size):
            original_img = images[i].float() / 255.0
            original_img = original_img.cpu().numpy()
            saliency_map = saliency_maps[i]
            
            output_path = os.path.join(model_output_dir, f"saliency_{sample_count:05d}.png")
            visualize_and_save_saliency(original_img, saliency_map, output_path)
            
            sample_count += 1
            if sample_count >= num_samples:
                break
        
        if sample_count >= num_samples:
            break
    
    print(f"✓ Generated {sample_count} saliency maps for {model_name}")
    print(f"  Saved to: {model_output_dir}")